# 向量加法

本教程将使用 Triton 编写一个简单的向量加法操作。

你将学习：

- Triton 的基本编程模型。
- 用于定义 Triton 内核的 `triton.jit` 装饰器。
- 如何对照原生参考实现验证自定义算子，并进行性能基准测试。

---

来源：[Triton 官方教程](https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html)，版本为 2026-10-16 下载的官网 main 文档。

中文练习版：保留说明与公式；原有代码单元及说明中的独立代码块均保留为空白代码框；移除代码、运行输出、测量结果与图表。空白框供自行填写，未运行教程。原文许可见 [TRITON-LICENSE.txt](TRITON-LICENSE.txt)。

## 计算内核

In [ ]:
import torch

import triton
import triton.language as tl

# 获取当前使用的 GPU, 并把它表示成 PyTorch 的设备对象, 保存到 DEVICE
DEVICE = triton.runtime.driver.active.get_active_torch_device()

@triton.jit
def add_kernel(
    x_ptr,  # 第一个 vector 指针
    y_ptr,  # 第二个 vector 指针
    output_ptr,  # 输出指针
    n_elements,  # vector 长度
    BLOCK_SIZE: tl.constexpr,  # 每个 program 实例处理多少个元素
    # NOTE: `constexpr` 会决定内核中张量的形状, 所以必须在编译时确定
    # 比如用 tl.arange(0, BLOCK_SIZE) 生成块内的元素编号,
    # 如果 BLOCK_SIZE=1024, 就会生成 0~1023编号, 形成 (1024,) triton 张量块
):
    # 向量长度 n_elements = 250, BLOCK_SIZE=64
    # 会启动 4 个 program 实例: 0, 1, 2, 3

    # 获取当前实例在启动网格第 0 个维度上的编号 (0/1/2/3)
    # 不同 program 实例得到不同的编号 (0/1/2/3)
    pid = tl.program_id(axis=0)
    # 不同 program 标号 * BLOCK_SIZE 得到不同 BLOCK 偏移
    # pid           block_start     负责的候选下标
    # 0                 0               0~63
    # 1                 64              64~127
    # 2                128              128~191
    # 3                192              192~255
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    # 64 * 4 = 256 > 250, 所以最后 6 个 mask 需要设置为 false
    mask =  offsets < n_elements
    # 通过 x_ptr/y_ptr + offsets 获取实际地址
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    output = x + y
    # Write x + y back to DRAM
    tl.store(output_ptr + offsets, output, mask=mask)


现在可以使用上述函数计算两个 PyTorch 张量的逐元素之和，并检验结果是否正确。

In [ ]:
def add(x: torch.Tensor, y: torch.Tensor):
    # 分配结果的存储空间
    # empty 表示不初始化元素值
    output = torch.empty_like(x)
    assert x.device == DEVICE and y.device == DEVICE and output.device == DEVICE
    # numel 返回所有维度合起来的元素数量, 比如 (20,30) 元素总数就是 20 x 30 = 600
    n_elements = output.numel()
    # 确定启动多少个 program 实例
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']), )
    # print(f"grid: {grid}")
    # add_kernel[grid] 为 triton 内核指定启动网格
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output

In [ ]:
torch.manual_seed(0)
size = 98432
x = torch.rand(size, device=DEVICE)
y = torch.rand(size, device=DEVICE)
output_torch = x + y
output_triton = add(x, y)
print(output_torch)
print(output_triton)
print(f'The maximum difference between torch and triton is '
      f'{torch.max(torch.abs(output_torch - output_triton))}')

完成正确性验证后，就可以继续进行性能测试。

## 性能基准测试

对长度逐渐增加的向量测试自定义算子，以了解它相对于 PyTorch 的性能表现。

Triton 提供了一组内置工具，方便绘制自定义算子在不同问题规模下的性能表现。

In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['size'],  # Argument names to use as an x-axis for the plot.
        x_vals=[2**i for i in range(12, 28, 1)],  # Different possible values for `x_name`.
        x_log=True,  # x axis is logarithmic.
        line_arg='provider',  # Argument name whose value corresponds to a different line in the plot.
        line_vals=['triton', 'torch'],  # Possible values for `line_arg`.
        line_names=['Triton', 'Torch'],  # Label name for the lines.
        styles=[('blue', '-'), ('green', '-')],  # Line styles.
        ylabel='GB/s',  # Label name for the y-axis.
        plot_name='vector-add-performance',  # Name for the plot. Used also as a file name for saving the plot.
        args={},  # Values for function arguments not in `x_names` and `y_name`.
    ))
def benchmark(size, provider):
    x = torch.rand(size, device=DEVICE, dtype=torch.float32)
    y = torch.rand(size, device=DEVICE, dtype=torch.float32)
    quantiles = [0.5, 0.2, 0.8]
    if provider == 'torch':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: x + y, quantiles=quantiles)
    if provider == 'triton':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: add(x, y), quantiles=quantiles)

    gbps = lambda ms: 3 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms), gbps(max_ms), gbps(min_ms)

现在可以运行经过装饰的基准测试函数。参数 `print_data` 控制是否打印性能数值，`show_plots` 控制是否显示图表，`save_path` 用于指定图表及原始 CSV 数据的保存目录。

In [ ]:
benchmark.run(print_data=True, show_plots=True)